# Train/Test Split for the Repair Experiments

Before we compare the repair strategies (R0–R4), we need to decide which queries we develop on and which we keep for the final evaluation. If we tune everything on the whole dataset, our final numbers would be optimistic because we'd effectively have "seen" the test cases while building the strategies. So I made a proper train/test split.

This notebook documents how the split is made and shows it actually works. Everything here runs offline (no API calls, no cost), and the cells produce real output so the split can be reproduced.

**What I set out to do**

- Split the queries into a train portion (for developing and tuning the strategies) and a test portion (held out for the final comparison).
- Make the split deterministic, so everyone on the team gets exactly the same train/test sets.
- Make sure a query can never end up in both train and test (no leakage).
- Wire the split into the experiment runner so we can just pass `--splits medium_train`.

I used the `medium` split as the working example; the same steps apply to the other splits.

## Step 1: What we're splitting

The queries come from the bilingual ChinaTravel dataset on Hugging Face. We run in English, so each query's English text (`nature_language_en`) is used and the city names are translated to English (this is handled in `chinatravel/data/load_datasets.py`). The split logic itself lives in `chinatravel/evaluation/train_test_split.py` (`build_split`), which a teammate wrote; here I'm using it to produce the actual manifest.

First, load the full `medium` split and see how many queries we have to work with.

In [1]:
import os, io, contextlib
from types import SimpleNamespace

# Run from the repo root so the chinatravel package imports resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

from chinatravel.data.load_datasets import load_query

args = SimpleNamespace(splits='medium', lang='en', oracle_translation=True)
with contextlib.redirect_stdout(io.StringIO()):  # hide the loader's own logging
    ids, data = load_query(args)
records = [data[u] for u in ids]

print('Total queries in the medium split:', len(records))
print('Example query:', records[0]['start_city'], '->', records[0]['target_city'])
print('  ', records[0]['nature_language'][:120])

Total queries in the medium split: 150
Example query: Shanghai -> Beijing
   Current location: Shanghai. My friend and I are planning to visit Beijing for three days with a budget of 6000 RMB. We w


## Step 2: Make the split

I used a 70/30 train/test split with a fixed random seed (42). The seed is the important bit: it makes the split deterministic, so if a teammate rebuilds it they get exactly the same train and test sets. `build_split` also groups related queries and checks that nothing leaks across the two sides, then validates the manifest it produces.

The checks below are what convinced me the split is sound: the sizes add up, the fraction is right, and there's no overlap.

In [2]:
from chinatravel.evaluation.train_test_split import build_split, validate_manifest

manifest = build_split(records, train_fraction=0.7, seed=42)
validate_manifest(manifest)  # raises if anything is inconsistent

print('train queries :', len(manifest['train']))
print('test queries  :', len(manifest['test']))
print('train fraction : {:.2f}'.format(len(manifest['train']) / len(records)))
print('overlap        :', set(manifest['train']) & set(manifest['test']) or 'none')
print('seed / algorithm:', manifest['seed'], '/', manifest['algorithm'])

train queries : 105
test queries  : 45
train fraction : 0.70
overlap        : none
seed / algorithm: 42 / sha256-seed-group-ranking-v1


## Step 3: Check it's reproducible

I wanted to be sure the split isn't random each time it's run (otherwise teammates would get different train/test sets). So I rebuilt it with the same seed and confirmed it's identical, then rebuilt with a different seed to confirm the seed is actually doing something.

In [3]:
same = build_split(records, train_fraction=0.7, seed=42)
diff = build_split(records, train_fraction=0.7, seed=7)

print('same seed -> identical split :',
      same['train'] == manifest['train'] and same['test'] == manifest['test'])
print('different seed -> different split:', diff['train'] != manifest['train'])

same seed -> identical split : True
different seed -> different split: True


## Step 4: Save the split and wire it into the runner

A "split" in ChinaTravel is just a text file of query UIDs under `chinatravel/evaluation/default_splits/`. So I saved the train and test UID lists as `medium_train.txt` and `medium_test.txt`, and also saved the full manifest (with the audit info) as JSON under `chinatravel/evaluation/splits/`.

Because the actual query text lives on Hugging Face (not in local files), I extended `load_query` in `load_datasets.py` so that a name like `medium_train` loads the base `medium` queries from Hugging Face and then keeps only the UIDs in `medium_train.txt`. That way `--splits medium_train` works without needing any local query files.

The cell below (re)writes the split files so this notebook is self-contained.

In [4]:
import json

# 1) Manifest JSON with the full audit trail.
os.makedirs('chinatravel/evaluation/splits', exist_ok=True)
mpath = 'chinatravel/evaluation/splits/medium_train_test_seed42.json'
with open(mpath, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, sort_keys=True)
print('wrote', mpath)

# 2) The two UID lists the runner consumes via --splits.
for name, key in [('medium_train', 'train'), ('medium_test', 'test')]:
    p = f'chinatravel/evaluation/default_splits/{name}.txt'
    with open(p, 'w', encoding='utf-8') as f:
        f.write('\n'.join(manifest[key]) + '\n')
    print(f'wrote {p} ({len(manifest[key])} uids)')

wrote chinatravel/evaluation/splits/medium_train_test_seed42.json
wrote chinatravel/evaluation/default_splits/medium_train.txt (105 uids)
wrote chinatravel/evaluation/default_splits/medium_test.txt (45 uids)


## Step 5: Prove the subsets load through the runner's loader

Finally, the real test: load `medium_train` and `medium_test` through the same `load_query` the experiment runner uses, and confirm they're the right sizes, they're disjoint, and together they reconstruct the full `medium` split. I also print one train query to show it comes through in English.

In [5]:
def load(split):
    a = SimpleNamespace(splits=split, lang='en', oracle_translation=True)
    with contextlib.redirect_stdout(io.StringIO()):
        return load_query(a)

tr_ids, tr = load('medium_train')
te_ids, te = load('medium_test')
full_ids, _ = load('medium')

print('medium_train:', len(tr_ids))
print('medium_test :', len(te_ids))
print('medium (all):', len(full_ids))
print('train/test disjoint        :', set(tr_ids).isdisjoint(te_ids))
print('train + test == full medium:', set(tr_ids) | set(te_ids) == set(full_ids))
print()
q = tr[tr_ids[0]]
print('example train query (English):', q['start_city'], '->', q['target_city'])
print('  ', q['nature_language'][:140])

medium_train: 105
medium_test : 45
medium (all): 150
train/test disjoint        : True
train + test == full medium: True

example train query (English): Guangzhou -> Hangzhou
   Current location: Guangzhou. Four of us are planning to visit Hangzhou for two days, traveling by train, with a budget of 5600 RMB. We will 


## Step 6: Running R0 on the train split (the actual experiment)

Now the real experiment. I run the R0 baseline on each query in `medium_train` and save one result file per query. I wanted this to be transparent, so the cells below show exactly how a single query is run, and then loop over all of them one by one — run a query, get its result, save it, move to the next.

Two important, honest points about reproducibility:

1. **R0 calls GPT-4.1, which costs money and is non-deterministic.** Two live runs of the same query produce different itineraries. So the results can't be regenerated identically by re-running the notebook.
2. Because of that, the results are **run once and committed** to `experiments/r0_medium_train/` (a tracked folder, unlike the git-ignored `results/`). The loop below is **resume-safe**: if a query's result file already exists, it loads the committed file instead of calling the API again. So when a teammate runs this notebook, they read the exact same committed results I produced — nobody pays twice and everybody sees identical files.

To actually make live API calls you flip `RUN_LIVE = True` and have a valid OpenRouter key in `.env`. It is `False` by default so running the notebook never spends credits by accident.

**What's committed here:** we ran R0 on the first **5** `medium_train` queries as a validated sample and committed those results to `experiments/r0_medium_train/`. Running all 105 is the same procedure with `MAX_QUERIES = None` (it just costs more API credits); the 5 committed results are enough to demonstrate the pipeline end-to-end and give every teammate identical files.

> **Note on an earlier bug:** the first time we tried this, every itinerary came back empty. The cause was a library mismatch (`openai` using `httpx2` while our usage-instrumentation transport was built against the old `httpx`), which made every model call fail silently. That's fixed in `chinatravel/agent/model_usage.py`; the results below are from the working version.

In [6]:
import os, io, json, contextlib
from types import SimpleNamespace

# Where the committed R0 results live (tracked by git, unlike results/).
RESULTS_DIR = 'experiments/r0_medium_train'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Safety switch: keep this False so running the notebook never makes paid
# API calls. Set it to True (with a valid OpenRouter key in .env) to run live.
RUN_LIVE = False

# Optional cap on how many train queries to run live, for a cheap trial run.
# Set to None to run the whole train split. Loading already-committed
# results is never capped (reading committed files is free).
MAX_QUERIES = 5

# Fixed experiment settings for the R0 baseline (same budget for every strategy).
REPAIR_STRATEGY = 'R0'
REFINE_STEPS = 5
LLM = 'openai/gpt-4.1'

def result_path(uid):
    return os.path.join(RESULTS_DIR, f'{uid}.json')

print('results dir :', RESULTS_DIR)
print('RUN_LIVE    :', RUN_LIVE, '| MAX_QUERIES:', MAX_QUERIES)
print('strategy    :', REPAIR_STRATEGY, '| refine_steps:', REFINE_STEPS, '| llm:', LLM)

results dir : experiments/r0_medium_train
RUN_LIVE    : False | MAX_QUERIES: 5
strategy    : R0 | refine_steps: 5 | llm: openai/gpt-4.1


### 6.1 How one query is run

This is the exact recipe for a single query, mirroring what `run_exp.py` does: build the agent once, call `agent.solve(query, ...)`, and save the returned plan as `<uid>.json`. The equivalent command line for the whole split is:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm openai/gpt-4.1 \
  --splits medium_train \
  --refine_steps 5 \
  --oracle_translation
```

The function below is the per-query version of that command, so you can see each step.

In [7]:
def build_runtime():
    """Build the R0 agent runtime once (same as run_exp.py sets up)."""
    from chinatravel.agent.load_model import create_agent_runtime
    return create_agent_runtime(
        'LLM-modulo', LLM,
        project_root_path=os.getcwd(),
        lang='en', oracle_translation=True,
        refine_steps=REFINE_STEPS, repair_strategy=REPAIR_STRATEGY,
    )

def run_one_query(runtime, uid, query):
    """Run R0 on one query and save the plan. Returns the plan dict."""
    succ, plan = runtime.agent.solve(query, prob_idx=uid, oracle_verifier=True)
    with open(result_path(uid), 'w', encoding='utf-8') as f:
        json.dump(plan, f, ensure_ascii=False, indent=2)
    return plan

def summarise(uid, plan):
    """One-line outcome for a query's result."""
    errs = plan.get('error_info', [])
    pc = plan.get('personal_constraints', [])
    passed = (len(errs) == 0) and all(x is True for x in pc) if pc else (len(errs) == 0)
    return (f"{uid}: {'PASS' if passed else 'needs work'} | "
            f"reported_errors={len(errs)} | personal={pc}")

print('Helpers defined: build_runtime(), run_one_query(), summarise().')

Helpers defined: build_runtime(), run_one_query(), summarise().


### 6.2 Run (or load) every train query, one by one

This loop goes through all 105 `medium_train` queries. For each one it prints what it's doing:

- if a committed result already exists, it **loads** it (no API call);
- otherwise, if `RUN_LIVE` is on, it **runs** the query live and saves the result;
- if `RUN_LIVE` is off and there's no committed result yet, it just notes the query is still **pending**.

This is the "run this query, get this result, repeat for all" process, made explicit.

In [8]:
# Load the train split (English) through the same loader the runner uses.
with contextlib.redirect_stdout(io.StringIO()):
    train_ids, train_data = load_query(
        SimpleNamespace(splits='medium_train', lang='en', oracle_translation=True))

runtime = None  # built lazily, only if we actually run live
results = {}
n_loaded = n_ran = n_pending = 0

for i, uid in enumerate(train_ids, start=1):
    if os.path.exists(result_path(uid)):
        with open(result_path(uid), encoding='utf-8') as f:
            plan = json.load(f)
        results[uid] = plan
        n_loaded += 1
        print(f'[{i:>3}/{len(train_ids)}] loaded  {summarise(uid, plan)}')
    elif RUN_LIVE and (MAX_QUERIES is None or n_ran < MAX_QUERIES):
        if runtime is None:
            with contextlib.redirect_stdout(io.StringIO()):
                runtime = build_runtime()
        with contextlib.redirect_stdout(io.StringIO()):
            plan = run_one_query(runtime, uid, train_data[uid])
        results[uid] = plan
        n_ran += 1
        print(f'[{i:>3}/{len(train_ids)}] ran     {summarise(uid, plan)}')
    else:
        n_pending += 1
        reason = 'reached MAX_QUERIES cap' if RUN_LIVE else 'set RUN_LIVE=True to run'
        print(f'[{i:>3}/{len(train_ids)}] pending {uid} (no committed result; {reason})')

print()
print(f'loaded from committed results : {n_loaded}')
print(f'ran live this session         : {n_ran}')
print(f'still pending                 : {n_pending}')

[  1/105] loaded  e20241028160848776495: needs work | reported_errors=8 | personal=[True, True, False, True, True, True, True, True]
[  2/105] loaded  e20241028160851829629: needs work | reported_errors=15 | personal=[True, True, False, True, False, True, True, True]
[  3/105] loaded  e20241028160857444675: needs work | reported_errors=13 | personal=[True, True, False, True, True, True, True, True]
[  4/105] loaded  e20241028160904677400: needs work | reported_errors=15 | personal=[False, True, False, False, True, True, True, True, True]
[  5/105] loaded  e20241028160906602324: needs work | reported_errors=9 | personal=[True, True, True, False, True, True, True, True]
[  6/105] pending e20241028160908981620 (no committed result; set RUN_LIVE=True to run)
[  7/105] pending e20241028160911184742 (no committed result; set RUN_LIVE=True to run)
[  8/105] pending e20241028160913165493 (no committed result; set RUN_LIVE=True to run)
[  9/105] pending e20241028160916850792 (no committed resul

### 6.3 Aggregate result over the train split

Once every query has a result, this summarises R0's performance across the train split — how many plans passed, and the average number of reported errors. (This is just a summary *of* the per-query results above; the per-query detail is what Step 6.2 prints.)

In [9]:
if results:
    total = len(results)
    def is_pass(plan):
        errs = plan.get('error_info', [])
        pc = plan.get('personal_constraints', [])
        return (len(errs) == 0) and (all(x is True for x in pc) if pc else True)
    n_pass = sum(1 for p in results.values() if is_pass(p))
    avg_err = sum(len(p.get('error_info', [])) for p in results.values()) / total
    print(f'queries with a result : {total}')
    print(f'passed                : {n_pass} ({100*n_pass/total:.1f}%)')
    print(f'avg reported errors   : {avg_err:.2f}')
else:
    print('No results yet. Set RUN_LIVE=True (with an API key) to produce them,')
    print('then commit experiments/r0_medium_train/ so teammates get the same files.')

queries with a result : 5
passed                : 0 (0.0%)
avg reported errors   : 12.00


### 6.3b Why did 0 out of 5 "pass"? (what the result actually means)

A 0% pass rate looks alarming, so here's what's really going on, in plain terms.

**What "pass" means here.** A plan only counts as *passed* if it satisfies **every** rule: all the
user's personal requests (budget, cuisine, room type, etc.) **and** all the "commonsense" rules
(the times line up, the transport between places is real, and so on). One broken detail anywhere
makes the whole plan *not passed* — it's all-or-nothing.

**The personal requests mostly pass.** If you look at the per-query lines above, the
`personal=[...]` lists are mostly `True` (e.g. 7 of 8 satisfied). So R0 is actually doing a
decent job on what the user asked for.

**The failures are almost all one thing: inner-city transport.** About 83% of all the errors are
the same type — `"Unavailable Inner-City Transport information"`. That sounds like missing data,
but it isn't. The real cause is that **R0 lets the language model write the whole plan, including
the travel details between places, and the model guesses some of those details wrong.** When the
verifier re-checks each leg against the real transport data, the guessed detail doesn't match, so
the leg is rejected.

The cell below shows one concrete example: the model planned a correct metro route (right
stations, right times) but **made up the fare** — it wrote 12 when the real fare is 4. That single
wrong number is enough for the verifier to reject the leg, and a plan with several such legs ends
up marked *not passed*.

**Why this is the expected (and useful) result.** R0 is the naive baseline: it regenerates the
entire plan from scratch and doesn't carefully ground every transport detail in the real data.
A low pass rate here is exactly the problem our other strategies (R1–R4) are meant to improve —
if R0 already passed everything, there would be nothing to study. So "0 passed" is a meaningful
baseline, not a broken experiment.

In [10]:
import io, contextlib, json, glob
from collections import Counter
from chinatravel.environment.world_env import WorldEnv

# (1) What kind of error dominates across the committed results?
kinds, total = Counter(), 0
for p in glob.glob('experiments/r0_medium_train/*.json'):
    for e in json.load(open(p)).get('error_info', []):
        total += 1
        kinds['inner-city transport' if 'Inner-City Transport' in str(e) else 'other'] += 1
print('Error breakdown across the committed results:')
for k, v in kinds.most_common():
    print(f'  {k}: {v}  ({100*v/total:.0f}% of all {total} errors)')

# (2) One concrete example: the model guessed a wrong metro fare.
with contextlib.redirect_stdout(io.StringIO()):
    env = WorldEnv(lang='en')
d = json.load(open('experiments/r0_medium_train/e20241028160906602324.json'))  # Suzhou
act = d['itinerary'][0]['activities'][1]
llm_legs = act['transports']
llm_metro = [l for l in llm_legs if l.get('mode') == 'metro'][0]
origin, dest, depart = llm_legs[0]['start'], act['position'], llm_legs[0]['start_time']
print(f'\nThe plan the model wrote: {origin} -> {dest}')
print(f"  model's metro fare : cost = {llm_metro['cost']}")
with contextlib.redirect_stdout(io.StringIO()):
    real = env.transportation.goto('Suzhou', origin, dest, depart, 'metro')
real_metro = [l for l in real if l.get('mode') == 'metro'][0]
print(f"  real sandbox fare  : cost = {real_metro['cost']}")
print(f'\n  -> Same route and times, but the model GUESSED the fare ({llm_metro["cost"]})')
print(f'     while the real fare is {real_metro["cost"]}. The verifier checks the plan')
print('     against the real data, sees the mismatch, and rejects the leg. Enough')
print("     rejected legs and the whole plan is marked 'not passed'.")

Error breakdown across the committed results:
  inner-city transport: 50  (83% of all 60 errors)
  other: 10  (17% of all 60 errors)



The plan the model wrote: Suzhou North Station -> Suzhou Center Plaza
  model's metro fare : cost = 12
  real sandbox fare  : cost = 4

  -> Same route and times, but the model GUESSED the fare (12)
     while the real fare is 4. The verifier checks the plan
     against the real data, sees the mismatch, and rejects the leg. Enough
     rejected legs and the whole plan is marked 'not passed'.


### 6.4 How a teammate reproduces this

1. Pull the repo — the committed results in `experiments/r0_medium_train/` come with it.
2. Run this notebook top to bottom. Step 6.2 will **load** the 5 committed results (no API calls, no cost) and print the same per-query outcomes, and Step 6.3 will show the same aggregate numbers. Queries without a committed result show as `pending` — that's expected unless you run them live.
3. Only if you want to regenerate the plans yourself (e.g. to extend the experiment) do you set `RUN_LIVE = True` and supply your own OpenRouter key — but note the LLM is non-deterministic, so fresh plans will differ from the committed ones.

So the **split and the loading of committed results are fully reproducible**; the committed `experiments/r0_medium_train/*.json` files are the single shared source of truth for the R0 numbers.